# 🏁 End-to-End Real-World ML Project
### Heart Disease Prediction — Putting It All Together

---

This notebook ties together the core concepts from the entire course into a single, realistic workflow.  
We will work through every major stage a practitioner faces:

| Stage | What we do |
|---|---|
| 1 | **Load & Explore** the UCI Heart Disease dataset |
| 2 | **Clean & Preprocess** — missing values, encoding, scaling |
| 3 | **Dimensionality Reduction** — PCA visualisation |
| 4 | **Train multiple models** — Logistic Regression, KNN, Decision Tree, Random Forest, SVM, Naive Bayes |
| 5 | **Evaluate & Compare** — accuracy, precision, recall, F1, ROC-AUC |
| 6 | **Handle class imbalance** — SMOTE |
| 7 | **Hyperparameter tuning** — GridSearchCV with nested CV |
| 8 | **Final model selection** and interpretation |

> 📌 **Dataset**: [UCI Heart Disease](https://archive.ics.uci.edu/ml/datasets/heart+disease) — loaded directly from `sklearn.datasets`.

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

# Classifiers (topics 01-07, 24)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Evaluation (topics 29, 63, 108)
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

# Class imbalance (topic 28)
from imblearn.over_sampling import SMOTE

print('All imports successful!')

## 2. Load & Explore the Dataset

We use the **Heart Disease** dataset (Cleveland subset, 303 patients, 13 features, binary target: disease present/absent).

In [ ]:
# Load via OpenML (heart-disease, id=43898)
heart = fetch_openml(name='heart-disease', version=1, as_frame=True, parser='auto')
df = heart.frame.copy()

# Binarise target: 0 = no disease, 1 = disease
df['target'] = (df['num'].astype(int) > 0).astype(int)
df.drop(columns=['num'], inplace=True)

print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### 2.1 Target Distribution (Class Imbalance check — Topic 27)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['target'].value_counts()
axes[0].bar(['No Disease (0)', 'Disease (1)'], counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Target Distribution')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=['No Disease', 'Disease'],
            autopct='%1.1f%%', colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Class Proportions')

plt.tight_layout()
plt.show()
print(counts)

### 2.2 Feature Correlation Heatmap

In [ ]:
# Encode categorical columns for correlation
df_encoded = df.copy()
for col in df_encoded.select_dtypes(include='category').columns:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col].astype(str))
df_encoded = df_encoded.apply(pd.to_numeric, errors='coerce')

plt.figure(figsize=(12, 8))
sns.heatmap(df_encoded.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Preprocessing

Concepts applied: feature encoding (Topic 57), feature scaling (Topic 01/24).

In [ ]:
# Separate features and target
X = df_encoded.drop(columns=['target'])
y = df_encoded['target']

# Handle missing values (if any) with median imputation
X = X.fillna(X.median())

print('Missing values after imputation:', X.isnull().sum().sum())
print('Feature matrix shape:', X.shape)
print('Target distribution:\n', y.value_counts())

In [ ]:
# Stratified train/test split (Topic 13 — Hold-out Method)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standard scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train set: {X_train_scaled.shape}, Test set: {X_test_scaled.shape}')

## 4. Dimensionality Reduction — PCA Visualisation (Topic 06)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train_scaled)

plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_train, cmap='bwr', alpha=0.7, edgecolors='k', linewidths=0.3)
plt.colorbar(scatter, label='Target (0=No Disease, 1=Disease)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('PCA — 2D projection of Heart Disease data')
plt.tight_layout()
plt.show()
print(f'Total variance explained by 2 PCs: {sum(pca.explained_variance_ratio_)*100:.1f}%')

## 5. Handle Class Imbalance with SMOTE (Topic 28)

In [ ]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

print('Before SMOTE:', pd.Series(y_train).value_counts().to_dict())
print('After  SMOTE:', pd.Series(y_train_res).value_counts().to_dict())

## 6. Train & Evaluate Multiple Models

We benchmark six classifiers (Topics 01–07, 24) using **5-fold stratified cross-validation** (Topic 11).

In [ ]:
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)':           SVC(kernel='rbf', probability=True, random_state=42),
    'Naive Bayes':         GaussianNB(),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, clf in classifiers.items():
    cv_scores = cross_val_score(clf, X_train_res, y_train_res, cv=cv, scoring='roc_auc')
    results.append({
        'Model': name,
        'CV ROC-AUC Mean': cv_scores.mean(),
        'CV ROC-AUC Std':  cv_scores.std(),
    })

results_df = pd.DataFrame(results).sort_values('CV ROC-AUC Mean', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
# Visual comparison
plt.figure(figsize=(9, 5))
plt.barh(
    results_df['Model'],
    results_df['CV ROC-AUC Mean'],
    xerr=results_df['CV ROC-AUC Std'],
    color='steelblue', edgecolor='black', alpha=0.8, capsize=4
)
plt.xlabel('ROC-AUC (5-Fold CV)')
plt.title('Model Comparison — Cross-Validated ROC-AUC')
plt.xlim(0.5, 1.0)
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning for the Best Model (Topic 14)

We tune **Random Forest** using `GridSearchCV` with nested cross-validation (Topic 12).

In [ ]:
param_grid = {
    'n_estimators':     [50, 100, 200],
    'max_depth':        [None, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_res, y_train_res)

print('Best params :', grid_search.best_params_)
print(f'Best CV AUC : {grid_search.best_score_:.4f}')

## 8. Final Evaluation on the Hold-Out Test Set (Topics 29, 108)

In [ ]:
best_clf = grid_search.best_estimator_
y_pred      = best_clf.predict(X_test_scaled)
y_pred_prob = best_clf.predict_proba(X_test_scaled)[:, 1]

print('=== Test Set Report ===')
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))
print(f'Test ROC-AUC: {roc_auc_score(y_test, y_pred_prob):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Confusion Matrix ---
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['No Disease', 'Disease'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Confusion Matrix')

# --- ROC Curve (Topic 29) ---
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
auc_score   = roc_auc_score(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, lw=2, label=f'Random Forest (AUC = {auc_score:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Feature Importance

Understanding *which* features the model relies on most is critical in practice.

In [ ]:
importances = pd.Series(best_clf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Random Forest — Feature Importances')
plt.xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

## 10. Summary & Key Takeaways

| Step | Concept applied | Notebook topics |
|---|---|---|
| EDA & Correlation | Data exploration | General |
| Hold-out split | Generalisation testing | 13 |
| Standard scaling | Feature preprocessing | 01, 24 |
| PCA visualisation | Dimensionality reduction | 06 |
| SMOTE oversampling | Class imbalance | 28 |
| Cross-validation | Model selection | 11 |
| GridSearchCV | Hyperparameter tuning | 14 |
| ROC/AUC curve | Evaluation | 29 |
| Confusion matrix / report | Classification metrics | 63, 108 |
| Feature importances | Model interpretability | 03, 04 |

---

### 💡 Next steps

- Try replacing Random Forest with an **XGBoost** or **LightGBM** model.
- Apply **SHAP values** for deeper interpretability.
- Build a simple **Streamlit** or **Gradio** app to serve the model.
- Swap in a different dataset (e.g., Titanic, Breast Cancer, Stroke Prediction) and see if your conclusions change.